# ML ver2

In [ ]:
import sys
sys.path.append('..')

In [2]:
import numpy as np
import pandas as pd

from sqlalchemy import create_engine
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.api import VAR


In [3]:
from dotenv import load_dotenv
import os

In [4]:

load_dotenv()

USER_NAME = os.getenv("USER_NAME")
PASSWORD = os.getenv("PASSWORD")
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = rf"postgresql+psycopg2://{USER_NAME}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)


In [5]:
sql = 'SELECT * FROM warehouse.vw_warehouse_sales; '
df = pd.read_sql(sql, con=engine)
print(df.shape)
df.head()

(7784, 17)


,sales_key,week,start_date,product,sales_units,cost_of_goods_sold_cogs,inventory_level,demand_forecast,price_per_unit,gross_profit,operating_profit,revenue,inventory_turnover_ratio,gross_profit_margin,revenue_growth,modified_row,run_id
0,1,1,2020-01-01,Rice - Basmati,1153.0,21.84,2915.0,1242.0,29.29,8589.85,5362.48,33771.37,0.28,0.2544,0.0000,False,20261002135808273756
1,2,2,2020-01-08,Rice - Basmati,1108.0,21.69,3307.0,1149.0,27.30,6215.88,3593.65,30248.40,0.38,0.2055,-0.1043,False,20261002135808273756
2,3,3,2020-01-15,Rice - Basmati,1527.0,22.38,3280.0,1639.0,27.48,7787.70,2713.53,41961.96,0.46,0.1856,0.3872,False,20261002135808273756
3,4,4,2020-01-22,Rice - Basmati,1299.0,21.41,1981.0,1093.0,26.56,6689.85,2756.27,34501.44,0.40,0.1939,-0.1778,False,20261002135808273756
4,5,5,2020-01-29,Rice - Basmati,1346.0,21.69,3135.0,1424.0,26.27,6164.68,1804.61,35359.42,0.68,0.1743,0.0249,False,20261002135808273756


In [6]:
df["week"] = pd.to_numeric(df["week"])
df["sales_units"] = pd.to_numeric(df["sales_units"])
df["demand_forecast"] = pd.to_numeric(df["demand_forecast"])
df["inventory_level"] = pd.to_numeric(df["inventory_level"])

df = df.sort_values(["week", "product"]).reset_index(drop=True)


# ============================================================
# 3. BASIC VALIDATION
# ============================================================

products = sorted(df["product"].unique())

print(f"Products: {len(products)}")
print(f"Weeks: {df['week'].min()} -> {df['week'].max()}")
print(f"Rows: {len(df):,}")

if len(products) != 30:
    print("WARNING: expected 30 products.")



Products: 30
Weeks: 1 -> 260
Rows: 7,784


In [7]:
# ============================================================
# 4. STOCKOUT IDENTIFICATION
# ============================================================
#
# A stockout is defined here as:
#
# inventory <= 0 AND sales_units == 0
#
# This does NOT prove lost demand. It is simply the operational
# definition we are using for adjustment.
# ============================================================

df["stockout"] = (
    (df["inventory_level"] <= 0) &
    (df["sales_units"] <= 0)
)

print("\nStockout observations:")
print(df["stockout"].value_counts())




Stockout observations:
stockout
False    7355
True      429
Name: count, dtype: int64


In [8]:
# ============================================================
# 5. PRODUCT MEDIANS
# ============================================================
#
# Used only as fallback when demand_forecast is unavailable.
# Calculate from non-stockout observations so the median isn't
# distorted by periods where inventory was zero.
# ============================================================

product_medians = (
    df.loc[~df["stockout"]]
      .groupby("product")["sales_units"]
      .median()
)



In [9]:
product_medians

product
Barley                   1500.0
Canned Tomatoes          1500.0
Canned Tuna              1500.0
Chickpeas                1027.5
Cocoa Powder             1531.0
Coffee - Arabica         1500.0
Coffee - Robusta         1402.0
Corn                     1500.0
Flour - All Purpose      1330.0
Lentils - Green          1500.0
Lentils - Red            1500.0
Milk Powder - Skimmed    1500.0
Milk Powder - Whole       804.0
Oats                     1500.0
Palm Oil                 1632.0
Pasta - Spaghetti        1500.0
Rice - Basmati           1209.5
Rice - White             1500.0
Salt - Iodized           1500.0
Soybean Oil              1500.0
Spices - Black Pepper    1597.0
Spices - Turmeric         570.0
Sugar - Raw Brown        1527.0
Sugar - White Refined    1500.0
Sunflower Oil             752.0
Tea - Black Ceylon       1500.0
Tea - Green              1500.0
Wheat - Hard Red          542.5
Wheat - Soft             1500.0
Yeast - Dry              1500.0
Name: sales_units, dtype: float6

In [10]:
df['demand_forecast'].min(), df['demand_forecast'].max()

(np.float64(297.0), np.float64(4910.0))

In [11]:
# df[stockout_with_forecast]
# df[stockout_without_forecast]
# df[(df['sales_units']==0) & (df['inventory_level']!=0)]

In [12]:
# ============================================================
# 6. CREATE STOCKOUT-ADJUSTED SALES
# ============================================================
#
# IMPORTANT:
# Actual sales remain untouched.
#
# For training:
#   stockout + demand_forecast available -> demand_forecast
#   stockout + demand_forecast missing    -> product median
#   otherwise                             -> actual sales
#
# This attempts to estimate underlying demand rather than
# observed constrained sales.
# ============================================================

df["sales_adjusted"] = df["sales_units"]

stockout_with_forecast = (
    df["stockout"] &
    df["demand_forecast"].notna()
)

df.loc[stockout_with_forecast, "sales_adjusted"] = (
    df.loc[stockout_with_forecast, "demand_forecast"]
)

stockout_without_forecast = (
    df["stockout"] &
    # df["demand_forecast"].isna()
    (df["modified_row"]==True)
)

df.loc[stockout_without_forecast, "sales_adjusted"] = (
    df.loc[stockout_without_forecast, "product"]
      .map(product_medians)
)

print(
    "\nStockouts replaced by demand forecast:",
    stockout_with_forecast.sum()
)

print(
    "Stockouts replaced by product median:",
    stockout_without_forecast.sum()
)



Stockouts replaced by demand forecast: 429
Stockouts replaced by product median: 31


In [13]:
df['sales_adjusted'].min(), df['sales_adjusted'].max()

(np.float64(0.0), np.float64(4599.0))

In [14]:
# ============================================================
# 7. TRAIN / TEST SPLIT
# ============================================================

TRAIN_END = 234
TEST_START = 235
TEST_END = 260

train_df = df[df["week"] <= TRAIN_END].copy()
test_df = df[
    (df["week"] >= TEST_START) &
    (df["week"] <= TEST_END)
].copy()



In [15]:
train_df['sales_adjusted'].min(), train_df['sales_adjusted'].max()

(np.float64(0.0), np.float64(4599.0))

In [16]:
# ============================================================
# 8. BUILD WEEK x PRODUCT MATRIX
# ============================================================
#
# Use adjusted sales for TRAINING only.
#
# For the future/test period we retain ACTUAL sales for
# evaluation.
# ============================================================

train_matrix = (
    train_df
    .pivot_table(
        index="week",
        columns="product",
        values="sales_adjusted",
        aggfunc="sum"
    )
    .reindex(columns=products)
    .sort_index()
)

test_matrix = (
    test_df
    .pivot_table(
        index="week",
        columns="product",
        values="sales_units",
        aggfunc="sum"
    )
    .reindex(columns=products)
    .sort_index()
)




In [17]:
train_matrix

product,Barley,Canned Tomatoes,Canned Tuna,Chickpeas,Cocoa Powder,Coffee - Arabica,Coffee - Robusta,Corn,Flour - All Purpose,Lentils - Green,...,Spices - Black Pepper,Spices - Turmeric,Sugar - Raw Brown,Sugar - White Refined,Sunflower Oil,Tea - Black Ceylon,Tea - Green,Wheat - Hard Red,Wheat - Soft,Yeast - Dry
week,,,,,,,,,,,,,,,,,,,,,
1,2669.0,2152.0,2722.0,1044.0,2080.0,2173.0,1538.0,2108.0,1198.0,2616.0,...,1966.0,553.0,1745.0,3012.0,732.0,2956.0,1644.0,541.0,2352.0,3147.0
2,2500.0,1971.0,1518.0,1017.0,1852.0,1800.0,1289.0,2236.0,1018.0,2779.0,...,265.0,579.0,1760.0,2523.0,809.0,1778.0,1746.0,534.0,2772.0,2052.0
3,1500.0,1701.0,2500.0,917.0,1858.0,2196.0,1682.0,314.0,1171.0,101.0,...,2207.0,590.0,1921.0,1500.0,677.0,2500.0,1713.0,617.0,1120.0,2500.0
4,0.0,2428.0,0.0,1156.0,1784.0,0.0,1739.0,1500.0,1391.0,2991.0,...,1793.0,553.0,2235.0,1500.0,790.0,2500.0,566.0,564.0,0.0,2500.0
5,1500.0,72.0,2500.0,1116.0,2056.0,2220.0,1482.0,0.0,1543.0,0.0,...,0.0,608.0,1808.0,2500.0,899.0,2500.0,1919.0,610.0,2500.0,1500.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230,1500.0,1500.0,1500.0,1215.0,1500.0,0.0,1500.0,2500.0,NaN,1500.0,...,2500.0,694.0,2202.0,0.0,860.0,2500.0,2074.0,654.0,1500.0,1500.0
231,0.0,1500.0,2500.0,1123.0,0.0,2500.0,1246.0,2500.0,1637.0,1500.0,...,1500.0,676.0,298.0,2500.0,938.0,2500.0,1736.0,797.0,2500.0,1500.0
232,1500.0,0.0,3615.0,1025.0,1500.0,2500.0,0.0,1500.0,1634.0,2500.0,...,1500.0,586.0,0.0,1500.0,956.0,1500.0,2081.0,613.0,0.0,1500.0


In [18]:
# Check completeness
if train_matrix.isna().any().any():
    print("\nWARNING: missing product/week combinations in training data.")

if test_matrix.isna().any().any():
    print("WARNING: missing product/week combinations in test data.")


In [19]:
# ============================================================
# 9. HANDLE MISSING PRODUCT-WEEK RECORDS
# ============================================================
#
# Only do this if an absent row really means zero sales in
# your warehouse. If an absent row means missing data, don't
# perform this fill.
# ============================================================

train_matrix = train_matrix.fillna(0)
test_matrix = test_matrix.fillna(0)


In [20]:
# ============================================================
# 10. WEEKLY TOTAL SALES
# ============================================================

train_total = train_matrix.sum(axis=1)
test_total = test_matrix.sum(axis=1)

if (train_total <= 0).any():
    zero_weeks = train_total[train_total <= 0].index.tolist()
    raise ValueError(
        f"Training contains weeks with zero total sales: {zero_weeks}"
    )


# ============================================================
# 11. CONVERT SALES -> WEEKLY PRODUCT SHARES
# ============================================================

train_shares = train_matrix.div(train_total, axis=0)

print("\nShare check:")
print(
    train_shares.sum(axis=1).describe()
)



Share check:
count    2.340000e+02
mean     1.000000e+00
std      1.627988e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64


In [21]:
print(
    train_shares.sum(axis=1).describe()
)


count    2.340000e+02
mean     1.000000e+00
std      1.627988e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64


In [22]:
# ============================================================
# 12. PCA
# ============================================================
#
# We reduce 30 product shares to a small number of latent
# components.
#
# 5-8 components is a sensible starting range.
# ============================================================

N_COMPONENTS = 6

pca = PCA(
    n_components=N_COMPONENTS,
    random_state=42
)

train_pca = pca.fit_transform(train_shares)

explained = pca.explained_variance_ratio_

print("\nPCA explained variance:")
for i, value in enumerate(explained, start=1):
    print(f"PC{i}: {value:.4f}")

print(
    f"Cumulative explained variance: "
    f"{explained.cumsum()[-1]:.4f}"
)




PCA explained variance:
PC1: 0.0928
PC2: 0.0850
PC3: 0.0713
PC4: 0.0679
PC5: 0.0616
PC6: 0.0570
Cumulative explained variance: 0.4355


In [23]:
# ============================================================
# 13. FORECAST PCA COMPONENTS WITH VAR
# ============================================================
#
# Each row = one week
# Each column = one PCA component.
#
# VAR captures relationships between the latent components.
# ============================================================

pca_df = pd.DataFrame(
    train_pca,
    index=train_shares.index,
    columns=[
        f"PC{i+1}"
        for i in range(N_COMPONENTS)
    ]
)

# Keep lag relatively small because we only have 234 weeks.
MAX_LAGS = 8

var_model = VAR(pca_df)

var_result = var_model.fit(
    maxlags=MAX_LAGS,
    ic="aic",
    trend="c"
)

print("\nSelected VAR lag:")
print(var_result.k_ar)

print("\nVAR summary:")
print(var_result.summary())




Selected VAR lag:
1

VAR summary:
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Sat, 03, Oct, 2026
Time:                     07:41:07
--------------------------------------------------------------------
No. of Equations:         6.00000    BIC:                   -42.8813
Nobs:                     233.000    HQIC:                  -43.2525
Log likelihood:           3126.47    FPE:                1.27869e-19
AIC:                     -43.5034    Det(Omega_mle):     1.07061e-19
--------------------------------------------------------------------
Results for equation PC1
            coefficient       std. error           t-stat            prob
-------------------------------------------------------------------------
const         -0.000050         0.001944           -0.026           0.980
L1.PC1        -0.231468         0.063810           -3.627           0.000
L1.PC2        -0.040844         0.066712           -0.6

c:\Users\msii\Desktop\work\interview\armani\armani_prj\venv_armani\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use on the supported classes of index.
  self._init_dates(dates, freq)


In [24]:
# ============================================================
# 14. FORECAST PCA COMPONENTS
# ============================================================

forecast_horizon = TEST_END - TRAIN_END

last_values = pca_df.values[-var_result.k_ar:]

pca_forecast = var_result.forecast(
    y=last_values,
    steps=forecast_horizon
)

forecast_weeks = np.arange(
    TEST_START,
    TEST_END + 1
)

pca_forecast_df = pd.DataFrame(
    pca_forecast,
    index=forecast_weeks,
    columns=pca_df.columns
)



In [25]:
# ============================================================
# 15. RECONSTRUCT PRODUCT SHARES
# ============================================================

forecast_shares_raw = pca.inverse_transform(
    pca_forecast
)

forecast_shares = pd.DataFrame(
    forecast_shares_raw,
    index=forecast_weeks,
    columns=products
)


# ============================================================
# 16. PROJECT BACK TO VALID SHARES
# ============================================================
#
# PCA reconstruction can produce:
#
#   negative shares
#   rows that don't sum exactly to 1
#
# We therefore:
#
#   1. clip negative values to zero
#   2. normalize each week to sum to one
#
# This is a practical projection back onto the simplex.
# ============================================================

forecast_shares = forecast_shares.clip(lower=0)

row_sums = forecast_shares.sum(axis=1)

# Protect against a pathological all-zero row.
for week in forecast_shares.index:
    if row_sums.loc[week] <= 0:
        forecast_shares.loc[week] = train_shares.mean(axis=0)

forecast_shares = forecast_shares.div(
    forecast_shares.sum(axis=1),
    axis=0
)


print("\nForecast share check:")
print(forecast_shares.sum(axis=1))




Forecast share check:
235    1.0
236    1.0
237    1.0
238    1.0
239    1.0
240    1.0
241    1.0
242    1.0
243    1.0
244    1.0
245    1.0
246    1.0
247    1.0
248    1.0
249    1.0
250    1.0
251    1.0
252    1.0
253    1.0
254    1.0
255    1.0
256    1.0
257    1.0
258    1.0
259    1.0
260    1.0
dtype: float64


In [26]:
train_total.index.values

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143,
       144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156,
       157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169,
       170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 18

In [27]:
print(train_total.index)
print(train_total.index.dtype)
print(type(train_total.index))

Index([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,
       ...
       225, 226, 227, 228, 229, 230, 231, 232, 233, 234],
      dtype='int64', name='week', length=234)
int64
<class 'pandas.Index'>


In [28]:
# 17. FORECAST TOTAL SALES

from statsmodels.tsa.ar_model import AutoReg
# Use a simple sequential index so statsmodels has a supported #
#  time-series index.

log_total = pd.Series( np.log1p(train_total.values), index=pd.RangeIndex(start=0, stop=len(train_total)) )

total_model = AutoReg( log_total, lags=13, trend="ct", seasonal=False ).fit()

# Forecast exactly 26 future observations
log_total_forecast = total_model.forecast( steps=forecast_horizon )
total_forecast = np.expm1( np.asarray(log_total_forecast) )
total_forecast = pd.Series( total_forecast, index=forecast_weeks, name="total_sales_forecast" )
# Make sure forecasts cannot become negative
total_forecast = total_forecast.clip(lower=0)

print("\nTotal sales forecast:") 
print(total_forecast)




Total sales forecast:
235    40629.871254
236    43220.075011
237    41374.443853
238    43258.937910
239    43905.693861
240    40924.681596
241    44013.461413
242    42851.146670
243    43432.726498
244    41847.028165
245    43352.862402
246    43794.405082
247    42007.435209
248    43507.999418
249    42869.496948
250    43300.141465
251    42703.944931
252    42986.583249
253    43369.300095
254    42794.886783
255    43065.367951
256    42926.306730
257    43269.317581
258    42867.160612
259    42969.460478
260    43208.988238
Name: total_sales_forecast, dtype: float64


In [29]:
# ============================================================
# 18. OPTIONAL: SEASONAL NAIVE TOTAL FORECAST
# ============================================================
#
# Useful as a sanity-check baseline.
# ============================================================

seasonal_total_forecast = pd.Series(
    [
        train_total.loc[week - 26]
        for week in forecast_weeks
    ],
    index=forecast_weeks,
    name="seasonal_naive_total"
)



In [30]:
# ============================================================
# 19. CONVERT SHARE FORECAST -> PRODUCT SALES FORECAST
# ============================================================

sales_forecast = forecast_shares.mul(
    total_forecast,
    axis=0
)

sales_forecast.columns = products

sales_forecast.index.name = "week"



In [31]:
# ============================================================
# 20. ROUND / NON-NEGATIVE
# ============================================================

sales_forecast = sales_forecast.clip(lower=0)


# ============================================================
# 21. ACTUAL TEST SALES
# ============================================================

actual_sales = test_matrix.loc[
    forecast_weeks,
    products
]


In [32]:
# ============================================================
# 22. EVALUATION
# ============================================================

y_true = actual_sales.values.flatten()
y_pred = sales_forecast.values.flatten()


rmse = np.sqrt(
    mean_squared_error(y_true, y_pred)
)

mae = mean_absolute_error(
    y_true,
    y_pred
)

smape = (
    100
    * np.mean(
        2 * np.abs(y_pred - y_true)
        /
        (
            np.abs(y_true) +
            np.abs(y_pred) +
            1e-8
        )
    )
)


print("\n====================================")
print("PCA SHARE MODEL RESULTS")
print("====================================")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"sMAPE: {smape:.2f}%")




PCA SHARE MODEL RESULTS
RMSE : 722.2066
MAE  : 484.8342
sMAPE: 46.64%


In [33]:
# ============================================================
# 23. COMPARE AGAINST DEMAND FORECAST
# ============================================================

demand_test = (
    test_df
    .pivot_table(
        index="week",
        columns="product",
        values="demand_forecast",
        aggfunc="sum"
    )
    .reindex(
        index=forecast_weeks,
        columns=products
    )
)

if not demand_test.isna().all().all():

    demand_true = actual_sales.values.flatten()
    demand_pred = demand_test.values.flatten()

    valid = np.isfinite(demand_pred)

    demand_rmse = np.sqrt(
        mean_squared_error(
            demand_true[valid],
            demand_pred[valid]
        )
    )

    demand_mae = mean_absolute_error(
        demand_true[valid],
        demand_pred[valid]
    )

    demand_smape = (
        100
        * np.mean(
            2 * np.abs(
                demand_pred[valid] -
                demand_true[valid]
            )
            /
            (
                np.abs(demand_true[valid]) +
                np.abs(demand_pred[valid]) +
                1e-8
            )
        )
    )

    print("\n====================================")
    print("DEMAND FORECAST BASELINE")
    print("====================================")
    print(f"RMSE : {demand_rmse:.4f}")
    print(f"MAE  : {demand_mae:.4f}")
    print(f"sMAPE: {demand_smape:.2f}%")
    print(f"Coverage: {valid.mean():.2%}")



DEMAND FORECAST BASELINE
RMSE : 1077.0797
MAE  : 682.3051
sMAPE: 49.51%
Coverage: 100.00%


In [34]:

# ============================================================
# 24. PRODUCT-LEVEL RESULTS
# ============================================================

product_results = []

for product in products:

    actual = actual_sales[product].values
    predicted = sales_forecast[product].values

    product_rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    product_mae = mean_absolute_error(
        actual,
        predicted
    )

    product_smape = (
        100
        * np.mean(
            2 * np.abs(predicted - actual)
            /
            (
                np.abs(actual) +
                np.abs(predicted) +
                1e-8
            )
        )
    )

    product_results.append({
        "product": product,
        "rmse": product_rmse,
        "mae": product_mae,
        "smape": product_smape
    })


product_results = pd.DataFrame(product_results)

product_results = product_results.sort_values(
    "rmse",
    ascending=False
)

print("\nProduct-level results:")
print(product_results.to_string(index=False))





Product-level results:


              product        rmse         mae      smape
         Wheat - Soft 1238.439665 1074.149429 126.634106
Sugar - White Refined 1105.773715  883.260709  87.698871
          Yeast - Dry 1073.366071  857.791027  84.217438
   Tea - Black Ceylon 1020.773560  797.585571  77.808963
          Canned Tuna  975.386274  782.317547  68.959886
    Pasta - Spaghetti  961.241835  731.130247  68.422885
       Salt - Iodized  934.362828  670.873781  71.785617
               Barley  911.405521  779.508041  65.816761
         Rice - White  878.016099  621.752639  66.367205
      Canned Tomatoes  853.985552  633.439416  66.020942
      Lentils - Green  838.902796  604.152631  52.620856
Milk Powder - Skimmed  832.494816  645.494088  56.849438
Spices - Black Pepper  816.840542  587.649677  57.094854
                 Corn  805.478128  588.382353  55.419349
     Coffee - Arabica  757.467180  537.286328  51.504600
          Soybean Oil  676.397666  570.643591  43.066057
                 Oats  617.761

In [35]:
# ============================================================
# 25. TOTAL SALES EVALUATION
# ============================================================

actual_total = test_total.loc[
    forecast_weeks
]

total_rmse = np.sqrt(
    mean_squared_error(
        actual_total,
        total_forecast
    )
)

seasonal_rmse = np.sqrt(
    mean_squared_error(
        actual_total,
        seasonal_total_forecast
    )
)

print("\n====================================")
print("TOTAL SALES FORECAST")
print("====================================")
print(f"AR total RMSE       : {total_rmse:.4f}")
print(f"Seasonal naive RMSE : {seasonal_rmse:.4f}")




TOTAL SALES FORECAST
AR total RMSE       : 6486.1445
Seasonal naive RMSE : 9922.0557


In [36]:
# ============================================================
# 26. SHARE FORECAST EVALUATION
# ============================================================

actual_test_total = actual_sales.sum(axis=1)

actual_test_shares = actual_sales.div(
    actual_test_total.replace(0, np.nan),
    axis=0
)

share_rmse = np.sqrt(
    np.mean(
        (
            actual_test_shares.values -
            forecast_shares.values
        ) ** 2
    )
)

print("\n====================================")
print("SHARE FORECAST")
print("====================================")
print(f"Share RMSE: {share_rmse:.8f}")




SHARE FORECAST
Share RMSE: 0.01873629


In [47]:
# ============================================================
# 27. SAVE LOCAL CSV FILES
# ============================================================

sales_forecast_long = (
    sales_forecast
    .reset_index()
    .melt(
        id_vars="week",
        var_name="product",
        value_name="predicted_sales"
    )
)

sales_forecast_long.to_csv(
    "pca_sales_forecast.csv",
    index=False
)

product_results.to_csv(
    "pca_product_results.csv",
    index=False
)

forecast_shares.reset_index().to_csv(
    "pca_forecast_shares.csv",
    index=False
)

print("\nSaved:")
print("  pca_sales_forecast.csv")
print("  pca_product_results.csv")
print("  pca_forecast_shares.csv")



Saved:
  pca_sales_forecast.csv
  pca_product_results.csv
  pca_forecast_shares.csv


In [38]:
sales_forecast_long = (
    sales_forecast
    .reset_index()
    .melt(
        id_vars="week",
        var_name="product",
        value_name="predicted_sales"
    )
)

In [39]:
sales_forecast_long

,week,product,predicted_sales
0,235,Barley,1488.582694
1,236,Barley,1564.904129
2,237,Barley,1493.801336
3,238,Barley,1566.449686
4,239,Barley,1587.751652
...,...,...,...
775,256,Yeast - Dry,1677.523669
776,257,Yeast - Dry,1690.928243
777,258,Yeast - Dry,1675.212289
778,259,Yeast - Dry,1679.210081
